# Práctica Grupo 3 — Autoencoder, UMAP y Clustering (Aprendizaje No Supervisado)
**Materia:** SIS-351 Machine Learning (3er Parcial)  
**Dataset:** Wine (`load_wine` de `scikit-learn`, 178 muestras, 13 variables fisicoquímicas)  
**Pregunta guía:** *¿Cómo influye la representación en los patrones que vemos?*

### Reglas metodológicas
- Se escalan todas las variables con `StandardScaler`.
- Se fija la semilla `random_state = 42` en UMAP y K-means.
- **No se utilizan las etiquetas reales de Wine** para escalar, proyectar ni elegir `k`.
- Las métricas externas (ARI/NMI) al final se declaran explícitamente como **comparación externa post-hoc**.

## 1. Instalación condicional (para Google Colab si falta `umap-learn`)

In [ ]:
import importlib.util
import sys

if importlib.util.find_spec("umap") is None:
    print("Instalando umap-learn==0.6.0...")
    !{sys.executable} -m pip install -q umap-learn==0.6.0 seaborn
else:
    print("umap-learn ya está instalado.")

## 2. Importaciones y configuración reproducible

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import umap
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
K_CLUSTERS = 3
KMEANS_N_INIT = 20
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTADOS_DIR = BASE_DIR / "resultados"
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

print(f"scikit-learn: {sklearn.__version__} | umap-learn: {umap.__version__}")

## 3. Carga de Wine y escalado con `StandardScaler`
Escalamos las 13 variables numéricas (media 0, desviación estándar 1) porque poseen escalas muy distintas (p. ej., `proline` ronda valores de ~1000 frente a `nonflavanoid_phenols` < 1.0).

In [ ]:
dataset = load_wine(as_frame=True)
X = dataset.data
y_externa = dataset.target  # Solo para comparación externa post-hoc

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

print(f"Dataset Wine: {X.shape[0]} muestras x {X.shape[1]} variables")
display(X.head())

## 4. Justificación no supervisada de `k` (sin usar etiquetas reales)
Evaluamos $k \in \{2, 3, 4, 5, 6\}$ mediante el método del codo (inercia) y el Silhouette Score sobre `X_escalado`.

In [ ]:
registros_k = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=KMEANS_N_INIT, random_state=RANDOM_STATE)
    etiquetas = km.fit_predict(X_escalado)
    registros_k.append({
        "k": k,
        "Inercia": round(float(km.inertia_), 4),
        "Silhouette_Espacio_Escalado": round(float(silhouette_score(X_escalado, etiquetas)), 4),
    })

df_k = pd.DataFrame(registros_k)
display(df_k)

## 5. Proyecciones 2D (PCA vs. UMAP) y K-means con el mismo `k = 3`

In [ ]:
# PCA 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z_pca = pca.fit_transform(X_escalado)

# UMAP 2D
reductor_umap = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    random_state=RANDOM_STATE,
)
Z_umap = reductor_umap.fit_transform(X_escalado)

# K-means con k=3 en cada representación
km_pca = KMeans(n_clusters=K_CLUSTERS, n_init=KMEANS_N_INIT, random_state=RANDOM_STATE).fit(Z_pca)
km_umap = KMeans(n_clusters=K_CLUSTERS, n_init=KMEANS_N_INIT, random_state=RANDOM_STATE).fit(Z_umap)

sil_pca = silhouette_score(Z_pca, km_pca.labels_)
sil_umap = silhouette_score(Z_umap, km_umap.labels_)

df_metricas = pd.DataFrame([
    {
        "Representacion": "PCA 2D (Lineal)",
        "k": K_CLUSTERS,
        "Silhouette_Score": round(float(sil_pca), 4),
        "ARI_Comparacion_Externa": round(float(adjusted_rand_score(y_externa, km_pca.labels_)), 4),
        "NMI_Comparacion_Externa": round(float(normalized_mutual_info_score(y_externa, km_pca.labels_)), 4),
    },
    {
        "Representacion": "UMAP 2D (No lineal)",
        "k": K_CLUSTERS,
        "Silhouette_Score": round(float(sil_umap), 4),
        "ARI_Comparacion_Externa": round(float(adjusted_rand_score(y_externa, km_umap.labels_)), 4),
        "NMI_Comparacion_Externa": round(float(normalized_mutual_info_score(y_externa, km_umap.labels_)), 4),
    },
])
display(df_metricas)

## 6. Visualización de Clusters en PCA 2D vs. UMAP 2D

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.0))
paleta = ["#2563eb", "#10b981", "#f59e0b"]

for cid in range(K_CLUSTERS):
    m1 = km_pca.labels_ == cid
    axes[0].scatter(Z_pca[m1, 0], Z_pca[m1, 1], s=48, alpha=0.85, color=paleta[cid], label=f"Cluster {cid}")
axes[0].scatter(km_pca.cluster_centers_[:, 0], km_pca.cluster_centers_[:, 1], s=180, c="#0f172a", marker="X", label="Centroides")
axes[0].set_title(f"K-means (k={K_CLUSTERS}) en PCA 2D\nSilhouette = {sil_pca:.4f}", fontweight="bold")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend()
axes[0].grid(alpha=0.2)

for cid in range(K_CLUSTERS):
    m2 = km_umap.labels_ == cid
    axes[1].scatter(Z_umap[m2, 0], Z_umap[m2, 1], s=48, alpha=0.85, color=paleta[cid], label=f"Cluster {cid}")
axes[1].scatter(km_umap.cluster_centers_[:, 0], km_umap.cluster_centers_[:, 1], s=180, c="#0f172a", marker="X", label="Centroides")
axes[1].set_title(f"K-means (k={K_CLUSTERS}) en UMAP 2D\nSilhouette = {sil_umap:.4f}", fontweight="bold")
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
fig.savefig(RESULTADOS_DIR / "clusters_pca_umap.png", dpi=300, bbox_inches="tight")
plt.show()